In [1]:
!pip install --upgrade "optree>=0.13.0" diffusers transformers accelerate sentencepiece protobuf bitsandbytes
!git clone https://github.com/Sainava/Sai-ToMA.git
!pip install ftfy

import sys
sys.path.append('/kaggle/working/Sai-ToMA')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.6/457.6 kB 12.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 80.2 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 115.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 104.2 MB/s eta 0:00:0000:010:01
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.7.0
    Uninstalling safetensors-0.7.0:
      Successfully uninstalled safetensors-0.7.0
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstall

In [2]:
%%writefile token_shape_capture_hook.py
import torch
from pathlib import Path

class TokenShapeCaptureHook:
    def __init__(self, block_name: str, save_dir: str):
        self.block_name = block_name
        self.save_dir = Path(save_dir)
        self.save_dir.mkdir(parents=True, exist_ok=True)
        self.call_count = 0
        self.log_file = self.save_dir / "shape_log.txt"

    def __call__(self, module, input, output):
        # New Diffusers FLUX single blocks return:
        # output = (encoder_hidden_states, hidden_states)
        if isinstance(output, tuple) and len(output) == 2:
            text_tensor = output[0]
            image_tensor = output[1]
        else:
            text_tensor = None
            image_tensor = output

        # Save image token matrix
        image_tensor_cpu = image_tensor.detach().cpu()
        image_matrix_path = self.save_dir / f"{self.block_name}_image_step_{self.call_count}.pt"
        torch.save(image_tensor_cpu, image_matrix_path)

        image_shape = list(image_tensor.shape)
        image_seq_len = image_tensor.shape[1] if len(image_tensor.shape) > 1 else None

        # Save text token matrix, if available
        if text_tensor is not None:
            text_tensor_cpu = text_tensor.detach().cpu()
            text_matrix_path = self.save_dir / f"{self.block_name}_text_step_{self.call_count}.pt"
            torch.save(text_tensor_cpu, text_matrix_path)

            text_shape = list(text_tensor.shape)
            text_seq_len = text_tensor.shape[1] if len(text_tensor.shape) > 1 else None
        else:
            text_matrix_path = None
            text_shape = None
            text_seq_len = None

        with open(self.log_file, "a") as f:
            f.write(
                f"block={self.block_name} | "
                f"call_count={self.call_count} | "
                f"image_shape={image_shape} | image_seq_len={image_seq_len} | "
                f"text_shape={text_shape} | text_seq_len={text_seq_len} | "
                f"image_matrix={image_matrix_path} | "
                f"text_matrix={text_matrix_path}\\n"
            )

        self.call_count += 1

Writing token_shape_capture_hook.py


In [3]:
%%writefile token_shape_capture_hook.py
import torch
import torch.nn as nn
from pathlib import Path

class TokenShapeCaptureHook:
    def __init__(self, block_name: str, save_dir: str):
        self.block_name = block_name
        self.save_dir = Path(save_dir)
        self.save_dir.mkdir(parents=True, exist_ok=True)
        self.call_count = 0
        self.log_file = self.save_dir / "shape_log.txt"

    def __call__(self, module, input, output):
        out_tensor = output[0] if isinstance(output, tuple) else output
        shape = out_tensor.shape
        seq_len = shape[1] if len(shape) > 1 else None

        torch.save(out_tensor.detach().cpu(), self.save_dir / f"{self.block_name}_step_{self.call_count}.pt")
        with open(self.log_file, "a") as f:
            f.write(f"call_count={self.call_count} | shape={list(shape)} | seq_len={seq_len}\n")
        self.call_count += 1

Overwriting token_shape_capture_hook.py


In [ ]:
import os
import torch
import shutil
from diffusers import FluxPipeline
from diffusers.quantizers import PipelineQuantizationConfig
from hidden_state_capture_hook import HiddenStateCaptureHook
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import logging

logging.getLogger("diffusers").setLevel(logging.ERROR)

def run_baseline():
    # 1. Securely fetch your Hugging Face token
    try:
        user_secrets = UserSecretsClient()
        hf_token = user_secrets.get_secret("HF_TOKEN")
    except Exception as e:
        print("Error: Could not find HF_TOKEN. Ensure the secret is attached to this notebook in the Add-ons menu!")
        return
    
    # 2. Globally login to Hugging Face (Prevents 401 Unauthorized errors)
    print("Authenticating with Hugging Face...")
    login(token=hf_token)
    
    model_id = "black-forest-labs/FLUX.1-schnell" 
    print(f"Loading Baseline {model_id}...")

    # Set up 4-bit quantization to fit FLUX in Kaggle's T4 GPUs
    quant_config = PipelineQuantizationConfig(
        quant_backend="bitsandbytes_4bit",
        quant_kwargs={"bnb_4bit_compute_dtype": torch.bfloat16, "bnb_4bit_quant_type": "nf4"}
    )
    
    # 3. Load the pipeline 
    pipe = FluxPipeline.from_pretrained(
        model_id, 
        quantization_config=quant_config, 
        torch_dtype=torch.bfloat16
    )
    
    # 4. Use CPU offload to manage memory safely with 4-bit quantization
    pipe.enable_model_cpu_offload()

    target_block_indices = [2, 10, 18]
    output_dir = "./flux_baseline_results"
    hooks, handles = [], []

    # 5. Attach the Hidden State Capture Hooks
    print("Attaching hooks...")
    for idx in target_block_indices:
        hook = HiddenStateCaptureHook(block_name=f"flux_block_{idx}", save_dir=output_dir)
        hooks.append(hook)
        target_block = getattr(pipe.transformer, 'single_transformer_blocks', pipe.transformer.transformer_blocks)[idx]
        handles.append(target_block.register_forward_hook(hook))
    
    prompts = [
        "A highly detailed portrait of a cyberpunk hacker in a neon-lit room, cinematic lighting, intricate facial features, ultra realistic",
        "A sweeping landscape of a distant alien planet with twin suns, dramatic mountains, reflective lakes, volumetric clouds, highly detailed",
        "A massive brutalist concrete building standing alone in a misty forest, architectural photography, sharp geometric lines, overcast lighting",
        "A close-up wildlife photograph of a magnificent snow leopard walking through fresh snow, ultra detailed fur, natural lighting, shallow depth of field",
        "A magical fantasy castle floating on a giant rock above the clouds, waterfalls cascading into the sky, flying dragons, golden sunset, highly detailed concept art"
    ]

    # 6. Generate and save the outputs
    for i, prompt in enumerate(prompts):
        print(f"\nProcessing Baseline Prompt {i+1}/3: '{prompt}'")
        
        # Catch the generated image in a variable
        image = pipe(prompt=prompt, num_inference_steps=10, guidance_scale=0.0).images[0]
        
        # Save the image as a PNG
        filename = f"flux_baseline_image_{i+1}.png"
        image.save(filename)
        print(f"✅ Saved {filename} to /kaggle/working/")
    
    # 7. Clean up and zip the captured tensors
    for handle in handles: handle.remove()
    shutil.make_archive("flux_baseline_results", 'zip', output_dir)
    print("\n✅ Baseline tensor zip created successfully! (flux_baseline_results.zip)")

run_baseline()

In [ ]:

    # Force Python and PyTorch to empty the garbage
import gc
gc.collect()
torch.cuda.empty_cache()
print("GPU Memory wiped and ready for the next run!")

In [ ]:
import os
import torch
import shutil
import logging
import types
import time
import csv
import numpy as np

from diffusers import FluxPipeline
from diffusers.quantizers import PipelineQuantizationConfig
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

from toma.patch import apply_patch
from toma.flux_scheduler import FluxScheduler

logging.getLogger("diffusers").setLevel(logging.ERROR)


def adapt_toma_single_blocks_for_new_diffusers(pipe):
    """
    Fixes ToMA FLUX single-block API mismatch with newer Diffusers.

    New Diffusers passes:
        hidden_states = image tokens
        encoder_hidden_states = text tokens

    Older ToMA expects:
        hidden_states = concatenated [text tokens, image tokens]

    This adapter concatenates text + image tokens before ToMA,
    then splits them back after ToMA.
    """

    for idx, block in enumerate(pipe.transformer.single_transformer_blocks):
        old_forward = block.forward

        def compatible_forward(
            self,
            hidden_states,
            encoder_hidden_states=None,
            temb=None,
            image_rotary_emb=None,
            joint_attention_kwargs=None,
            *args,
            _old_forward=old_forward,
            _idx=idx,
            **kwargs,
        ):
            if encoder_hidden_states is None:
                try:
                    return _old_forward(
                        hidden_states=hidden_states,
                        temb=temb,
                        image_rotary_emb=image_rotary_emb,
                        joint_attention_kwargs=joint_attention_kwargs,
                    )
                except TypeError:
                    return _old_forward(
                        hidden_states,
                        temb,
                        image_rotary_emb,
                        joint_attention_kwargs,
                    )

            text_seq_len = encoder_hidden_states.shape[1]

            combined_hidden_states = torch.cat(
                [encoder_hidden_states, hidden_states],
                dim=1,
            )

            try:
                out = _old_forward(
                    hidden_states=combined_hidden_states,
                    temb=temb,
                    image_rotary_emb=image_rotary_emb,
                    joint_attention_kwargs=joint_attention_kwargs,
                )
            except TypeError:
                out = _old_forward(
                    combined_hidden_states,
                    temb,
                    image_rotary_emb,
                    joint_attention_kwargs,
                )

            if isinstance(out, tuple):
                if len(out) == 2:
                    return out
                out = out[0]

            new_encoder_hidden_states = out[:, :text_seq_len]
            new_hidden_states = out[:, text_seq_len:]

            return new_encoder_hidden_states, new_hidden_states

        block.forward = types.MethodType(compatible_forward, block)

    print("✅ Adapted ToMA single_transformer_blocks for new Diffusers API")


def find_flux_config():
    possible_config_paths = [
        "./config/flux.yaml",
        "/kaggle/working/Sai-ToMA/config/flux.yaml",
    ]

    for path in possible_config_paths:
        if os.path.exists(path):
            return path

    raise FileNotFoundError(
        "Could not find flux.yaml. Expected one of: "
        "./config/flux.yaml or /kaggle/working/Sai-ToMA/config/flux.yaml"
    )

def make_flux_scheduler(num_inference_steps, config_path):
    recompute_step = list(range(0, num_inference_steps))
    merge_step = list(range(0, num_inference_steps, 3))

    scheduler = FluxScheduler(
        timesteps=num_inference_steps,
        dst_recompute_timesteps=recompute_step,
        attn_recompute_timesteps=recompute_step,
        merge_step=merge_step,
        config_path=config_path,
    )

    old_step_for_image = scheduler.step_for_image
    old_step_for_text = scheduler.step_for_text

    def safe_step_for_image():
        out = old_step_for_image()
        if isinstance(out, tuple):
            return out
        return False, True

    def safe_step_for_text():
        out = old_step_for_text()
        if isinstance(out, tuple):
            return out
        return False, True

    scheduler.step_for_image = safe_step_for_image
    scheduler.step_for_text = safe_step_for_text

    return scheduler


def inject_scheduler_into_toma(pipe, scheduler):
    """
    ToMA stores merge_scheduler inside _toma_info.
    Before every prompt, inject a fresh scheduler so it is not exhausted.
    """
    count = 0
    for module in pipe.transformer.modules():
        if hasattr(module, "_toma_info") and isinstance(module._toma_info, dict):
            module._toma_info["merge_scheduler"] = scheduler
            count += 1
        if hasattr(module, "processor"):
            processor = module.processor
            if hasattr(processor, "_toma_info") and isinstance(processor._toma_info, dict):
                processor._toma_info["merge_scheduler"] = scheduler
                count += 1
    print(f"✅ Fresh scheduler injected into {count} ToMA locations")


def run_official_toma():
    # -----------------------------
    # 1. Hugging Face authentication
    # -----------------------------
    try:
        user_secrets = UserSecretsClient()
        hf_token = user_secrets.get_secret("HF_TOKEN")
    except Exception:
        print("Error: Could not find HF_TOKEN. Ensure the secret is attached in Kaggle!")
        return

    print("Authenticating with Hugging Face...")
    login(token=hf_token)

    # -----------------------------
    # 2. Settings (Matched exactly to run_flux.py)
    # -----------------------------
    model_id = "black-forest-labs/FLUX.1-schnell"

    num_inference_steps = 10
    ratio = 0.5
    dst_selection = "local_tile_wise_facility"
    num_tiles = 256
    merge_method = "attention"
    height = 512
    width = 512
    guidance_scale = 7.5

    output_dir = "official_toma_images"
    csv_file = "official_toma_results.csv"

    config_path = find_flux_config()
    print(f"Using ToMA config: {config_path}")

    # -----------------------------
    # 3. Load FLUX pipeline
    # -----------------------------
    print(f"Loading model: {model_id}")

    quant_config = PipelineQuantizationConfig(
        quant_backend="bitsandbytes_4bit",
        quant_kwargs={
            "bnb_4bit_compute_dtype": torch.bfloat16,
            "bnb_4bit_quant_type": "nf4",
        },
    )

    pipe = FluxPipeline.from_pretrained(
        model_id,
        quantization_config=quant_config,
        torch_dtype=torch.bfloat16,
    )

    # -----------------------------
    # 4. Apply ToMA patch once
    # -----------------------------
    print("Applying ToMA Patch...")

    initial_scheduler = make_flux_scheduler(num_inference_steps, config_path)

    apply_patch(
        model=pipe,
        ratio=ratio,
        dst_selection=dst_selection,
        num_tiles=num_tiles,
        merge_method=merge_method,
        merge_scheduler=initial_scheduler,
    )

    adapt_toma_single_blocks_for_new_diffusers(pipe)

    print("✅ ToMA patch applied successfully")

    # -----------------------------
    # 5. Offload model safely
    # -----------------------------
    pipe.enable_model_cpu_offload()

    # -----------------------------
    # 6. Prepare output folder & CSV
    # -----------------------------
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir, exist_ok=True)
    
    with open(csv_file, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(["Prompt", "Seed", "Inference Time (seconds)"])

    # -----------------------------
    # 7. Prompts
    # -----------------------------
    prompts = [
        "A highly detailed portrait of a cyberpunk hacker in a neon-lit room, cinematic lighting, intricate facial features, ultra realistic",
        "A sweeping landscape of a distant alien planet with twin suns, dramatic mountains, reflective lakes, volumetric clouds, highly detailed",
        "A massive brutalist concrete building standing alone in a misty forest, architectural photography, sharp geometric lines, overcast lighting",
        "A close-up wildlife photograph of a magnificent snow leopard walking through fresh snow, ultra detailed fur, natural lighting, shallow depth of field",
        "A magical fantasy castle floating on a giant rock above the clouds, waterfalls cascading into the sky, flying dragons, golden sunset, highly detailed concept art"
    ]

    inference_times = []
    total_start_time = time.perf_counter()

    # -----------------------------
    # 8. Run inference & Runtime measurements
    # -----------------------------
    for i, prompt in enumerate(prompts):
        print(f"\nToMA prompt {i + 1}/{len(prompts)}: {prompt}")

        fresh_scheduler = make_flux_scheduler(num_inference_steps, config_path)
        inject_scheduler_into_toma(pipe, fresh_scheduler)

        # Uses the identical seed for all prompts, matching official repo args.seed
        generator = torch.Generator(device="cpu").manual_seed(seed)

        start_time = time.perf_counter()

        result = pipe(
            prompt=prompt,
            height=height,
            width=width,
            num_inference_steps=num_inference_steps,
            guidance_scale=guidance_scale,
            max_sequence_length=512,
            generator=generator,
            output_type="pil",
        )

        end_time = time.perf_counter()
        inference_time = end_time - start_time
        inference_times.append(inference_time)

        image = result.images[0]
        image_path = os.path.join(output_dir, f"toma_prompt_{i + 1}.png")
        image.save(image_path)

        # Log into CSV
        with open(csv_file, mode='a', newline='') as file:
            writer = csv.writer(file)
            writer.writerow([prompt, seed, inference_time])

        print(f"✅ Image saved: {image_path}")
        print(f"⏱️ Inference time: {inference_time:.2f} seconds")

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # -----------------------------
    # 9. Compute stats
    # -----------------------------
    total_time = time.perf_counter() - total_start_time
    avg_time = np.mean(inference_times)
    std_time = np.std(inference_times)

    print("\n" + "="*40)
    print("📈 ToMA Inference Runtime Statistics")
    print("="*40)
    print(f"Total Runtime:     {total_time:.2f} seconds")
    print(f"Average Inference: {avg_time:.2f} seconds")
    print(f"Standard Dev:      {std_time:.2f} seconds")
    print("CSV saved to:      official_toma_results.csv")
    print("Images saved in:   official_toma_images/")
    print("="*40)

run_official_toma()

In [ ]:
import os
import torch
import torch.nn as nn
import shutil
import math
import time
import csv
import json
import logging

from diffusers import FluxPipeline
from diffusers.quantizers import PipelineQuantizationConfig
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

logging.getLogger("diffusers").setLevel(logging.ERROR)
logger = logging.getLogger(__name__)

# ==============================================================================
# 1. OFFICIAL FASTCACHE IMPLEMENTATION 
# (Reconstructed from xfuser/model_executor/accelerator/fastcache.py)
# ==============================================================================

class FastCacheAccelerator(nn.Module):
    """
    FastCache: A hidden-state-level caching and compression framework
    for accelerating DiT inference by eliminating redundant computations.
    """
    
    def __init__(
        self, 
        model: nn.Module,
        cache_ratio_threshold: float = 0.05,
        motion_threshold: float = 0.1,
        significance_level: float = 0.05,
        cache_enabled: bool = True
    ):
        super().__init__()
        self.model = model
        self.cache_ratio_threshold = cache_ratio_threshold
        self.motion_threshold = motion_threshold
        self.significance_level = significance_level
        self.cache_enabled = cache_enabled
        
        # Cache states
        self.prev_hidden_states = None
        self.bg_hidden_states = None
        self.cache_hits = 0
        self.total_steps = 0
        
        # Initialize cache adaptation parameters
        self.beta0 = 0.01
        self.beta1 = 0.5
        self.beta2 = -0.002
        self.beta3 = 0.00005
        
        # For statistics
        self.layer_cache_hits = {}
        
        # Linear approximation for static tokens
        if hasattr(model, "config") and hasattr(model.config, "hidden_size"):
            hidden_size = model.config.hidden_size
        else:
            hidden_size = 3072 # fallback for FLUX
            
        self.cache_projection = nn.Linear(hidden_size, hidden_size)
        
        logger.info(f"Initialized FastCache with thresholds: cache={cache_ratio_threshold}, motion={motion_threshold}")
    
    def get_adaptive_threshold(self, variance_score, timestep):
        """Calculate adaptive threshold based on variance and timestep"""
        normalized_timestep = timestep / 1000.0  # Normalize timestep to [0,1] range
        return (self.beta0 + 
                self.beta1 * variance_score + 
                self.beta2 * normalized_timestep + 
                self.beta3 * normalized_timestep**2)
    
    def compute_relative_change(self, current, previous):
        """Compute relative change between current and previous hidden states"""
        if previous is None:
            return float('inf')
            
        # Compute Frobenius norm of difference
        diff_norm = torch.norm(current - previous, p='fro')
        prev_norm = torch.norm(previous, p='fro')
        
        # Avoid division by zero
        if prev_norm == 0:
            return float('inf')
            
        return (diff_norm / prev_norm).item()
    
    def should_use_cache(self, hidden_states, timestep):
        """Determine if cached states should be used based on statistical test"""
        if not self.cache_enabled or self.prev_hidden_states is None:
            return False
            
        # Compute relative change
        delta = self.compute_relative_change(hidden_states, self.prev_hidden_states)
        
        # Compute threshold based on chi-square distribution
        n, d = hidden_states.shape[1], hidden_states.shape[2]  # token count, hidden dim
        dof = n * d  # degrees of freedom
        
        # Chi-square threshold for given significance level
        # Approximate chi-square using normal distribution for large DOF
        z = 1.96  # z-score for 95% confidence (significance_level=0.05)
        chi2_threshold = dof + z * math.sqrt(2 * dof)
        statistical_threshold = math.sqrt(chi2_threshold / dof)
        
        # Adaptive threshold based on timestep
        adaptive_threshold = self.get_adaptive_threshold(delta, timestep)
        
        # Final threshold combines both statistical and adaptive thresholds
        # We use both to ensure both statistical validity and context-specific adaptation
        final_threshold = max(self.cache_ratio_threshold, 
                              min(statistical_threshold, adaptive_threshold))
        
        return delta <= final_threshold
    
    def compute_motion_saliency(self, hidden_states):
        """Compute motion saliency for each token"""
        if self.prev_hidden_states is None:
            return torch.ones(hidden_states.shape[1], device=hidden_states.device)
            
        # Compute token-wise differences
        token_diffs = (hidden_states - self.prev_hidden_states).abs()
        
        # Take max across feature dimension to get token saliency
        token_saliency = token_diffs.max(dim=-1)[0].squeeze(0)
        
        # Normalize saliency
        if token_saliency.max() > 0:
            token_saliency = token_saliency / token_saliency.max()
            
        return token_saliency
    
    def forward(self, hidden_states, timestep=None, use_cached_states=True, layer_idx=None, **kwargs):
        """
        Process hidden states through FastCache
        """
        self.total_steps += 1
        
        # If caching is disabled or it's the first step, execute normally
        if not self.cache_enabled or self.prev_hidden_states is None:
            if layer_idx is not None and layer_idx not in self.layer_cache_hits:
                self.layer_cache_hits[layer_idx] = 0
                
            output = self.model(hidden_states, **kwargs)
            self.prev_hidden_states = hidden_states.detach().clone()
            if self.bg_hidden_states is None:
                self.bg_hidden_states = hidden_states.detach().clone()
            return output
        
        # Analyze motion and determine caching strategy
        if use_cached_states and self.should_use_cache(hidden_states, timestep):
            # Cache hit - reuse previous states
            self.cache_hits += 1
            if layer_idx is not None:
                self.layer_cache_hits[layer_idx] = self.layer_cache_hits.get(layer_idx, 0) + 1
                
            # Apply linear projection instead of full transformer
            output = self.cache_projection(hidden_states)
            return output
        
        # Compute motion saliency for tokens
        motion_saliency = self.compute_motion_saliency(hidden_states)
        motion_mask = motion_saliency > self.motion_threshold
        
        # If significant motion is detected, process normally
        if motion_mask.sum() / motion_mask.numel() > 0.5:
            output = self.model(hidden_states, **kwargs)
        else:
            # Split tokens into motion and static tokens
            batch_size, seq_len, hidden_dim = hidden_states.shape
            motion_indices = torch.where(motion_mask)[0]
            static_indices = torch.where(~motion_mask)[0]
            
            if len(motion_indices) > 0:
                # Process motion tokens through full transformer
                motion_states = hidden_states.index_select(1, motion_indices)
                motion_output = self.model(motion_states, **kwargs)
                
                # Process static tokens through linear projection
                static_states = hidden_states.index_select(1, static_indices)
                static_output = self.cache_projection(static_states)
                
                # Merge outputs
                output = hidden_states.clone()
                output.index_copy_(1, motion_indices, motion_output)
                output.index_copy_(1, static_indices, static_output)
            else:
                # All tokens are static, use linear projection
                output = self.cache_projection(hidden_states)
        
        # Update cache
        self.prev_hidden_states = hidden_states.detach().clone()
        # Update background state with exponential moving average
        alpha = 0.9
        self.bg_hidden_states = alpha * self.bg_hidden_states + (1 - alpha) * hidden_states.detach().clone()
        
        return output

# ==============================================================================
# 2. Adapters for Newer Diffusers FLUX Pipeline API
# ==============================================================================
# Global variable to intercept the timestep so FastCache knows the current step
CURRENT_TIMESTEP = None

class FluxTransformerTimestepTracker(nn.Module):
    """
    Wraps FluxTransformer2DModel to track the timestep without altering its behaviour.
    """
    def __init__(self, transformer):
        super().__init__()
        self.transformer = transformer
        
    def forward(self, hidden_states, timestep=None, **kwargs):
        global CURRENT_TIMESTEP
        
        # In diffusers, FLUX timesteps might be float tensors around 1.0 -> 0.0 or 1000 -> 0
        if isinstance(timestep, torch.Tensor):
            t_val = timestep.item() if timestep.numel() == 1 else timestep[0].item()
        else:
            t_val = timestep
            
        # FastCache expects a normalized scale where t_val / 1000 normalizes it to [0,1].
        # If diffusers is already passing 1.0 (or below), scale it to 1000 so FastCache logic works out of the box.
        if t_val is not None and t_val <= 1.0:
            CURRENT_TIMESTEP = t_val * 1000.0
        else:
            CURRENT_TIMESTEP = t_val
            
        return self.transformer(hidden_states, timestep=timestep, **kwargs)
        
    def __getattr__(self, name):
        return getattr(self.transformer, name)


class DiffusersFluxBlockAdapter(nn.Module):
    """
    Adapts new Diffusers FLUX blocks (which take separate text and image inputs and return tuples)
    into the concatenated-input format that FastCache mathematically relies upon.
    """
    def __init__(self, block):
        super().__init__()
        self.block = block
        
        class FakeConfig: pass
        self.config = FakeConfig()
        if hasattr(block, "hidden_size"):
            self.config.hidden_size = block.hidden_size
        elif hasattr(block, "dim"):
            self.config.hidden_size = block.dim
        else:
            self.config.hidden_size = 3072 # FLUX fallback
            
    def forward(self, hidden_states, **kwargs):
        # Unpack the FastCache concatenated tensor back into separate diffusers inputs
        text_seq_len = kwargs.pop("text_seq_len", 512)
        encoder_hidden_states = hidden_states[:, :text_seq_len]
        image_hidden_states = hidden_states[:, text_seq_len:]
        
        out = self.block(
            hidden_states=image_hidden_states,
            encoder_hidden_states=encoder_hidden_states,
            **kwargs
        )
        
        # Repackage output tuples into a single tensor for FastCache
        if isinstance(out, tuple):
            if len(out) == 2:
                # new diffusers returns (encoder_hidden_states, hidden_states) or vice versa depending on block
                if out[0].shape[1] == encoder_hidden_states.shape[1]:
                    return torch.cat([out[0], out[1]], dim=1)
                else:
                    return torch.cat([out[1], out[0]], dim=1)
            else:
                return out[0]
        return out


class FastCacheOuterWrapper(nn.Module):
    """
    Wraps the whole module combination. Bridges Diffusers separation logic 
    safely around the intact FastCacheAccelerator.
    """
    def __init__(self, diffusers_block, cache_threshold=0.05, motion_threshold=0.1):
        super().__init__()
        
        # Inner adapter to handle splitting and tuple returns for Diffusers
        self.inner_adapter = DiffusersFluxBlockAdapter(diffusers_block)
        
        # The authentic FastCache accelerator 
        self.fastcache = FastCacheAccelerator(
            self.inner_adapter,
            cache_ratio_threshold=cache_threshold,
            motion_threshold=motion_threshold
        )
        
    def forward(self, hidden_states, encoder_hidden_states=None, **kwargs):
        if encoder_hidden_states is None:
            # Fallback
            return self.fastcache(hidden_states, timestep=CURRENT_TIMESTEP, **kwargs)
            
        text_seq_len = encoder_hidden_states.shape[1]
        
        # Combine text and image tokens so FastCache calculates full context motion accurately
        combined = torch.cat([encoder_hidden_states, hidden_states], dim=1)
        
        # Call the pristine FastCache with tracked timestep
        kwargs["text_seq_len"] = text_seq_len
        out = self.fastcache(combined, timestep=CURRENT_TIMESTEP, **kwargs)
        
        # Split back out to Diffusers spec
        new_encoder_hidden_states = out[:, :text_seq_len]
        new_hidden_states = out[:, text_seq_len:]
        
        # Determine the correct tuple order to return based on the block type
        if hasattr(self.inner_adapter.block, "ff_context"): 
            # FluxTransformerBlock (double block) returns (hidden, encoder)
            return new_hidden_states, new_encoder_hidden_states
        else:
            # FluxSingleTransformerBlock (single block) returns (encoder, hidden)
            return new_encoder_hidden_states, new_hidden_states

# ==============================================================================
# 3. Main Execution Function
# ==============================================================================

def run_official_fastcache():
    # -----------------------------
    # A. Auth & Model Setup
    # -----------------------------
    try:
        user_secrets = UserSecretsClient()
        hf_token = user_secrets.get_secret("HF_TOKEN")
    except Exception:
        print("Error: Could not find HF_TOKEN. Ensure the secret is attached in Kaggle!")
        return

    print("Authenticating with Hugging Face...")
    login(token=hf_token)

    model_id = "black-forest-labs/FLUX.1-schnell"

    # Settings consistent with baseline
    num_inference_steps = 10
    height = 512
    width = 512
    guidance_scale = 0.0

    # FastCache specific settings 
    cache_ratio_threshold = 0.05
    motion_threshold = 0.1

    output_dir = "official_fastcache_images"
    csv_file = "official_fastcache_results.csv"

    print(f"Loading FastCache model: {model_id}")

    # Use BitsAndBytes Quantization identically to Baseline
    quant_config = PipelineQuantizationConfig(
        quant_backend="bitsandbytes_4bit",
        quant_kwargs={
            "bnb_4bit_compute_dtype": torch.bfloat16,
            "bnb_4bit_quant_type": "nf4",
        },
    )

    pipe = FluxPipeline.from_pretrained(
        model_id,
        quantization_config=quant_config,
        torch_dtype=torch.bfloat16,
    )

    # -----------------------------
    # B. Inject FastCache Logic
    # -----------------------------
    print("Applying FastCache Adapter Wrappers...")
    
    # 1. Track timestep at UNet level
    pipe.transformer = FluxTransformerTimestepTracker(pipe.transformer)
    
    # 2. Recursively apply FastCache to all Transformer blocks (matches official repository)
    wrapped_blocks = []
    
    def apply_fastcache_recursively(module, prefix=''):
        for name, child in module.named_children():
            full_name = f"{prefix}.{name}" if prefix else name
            module_type = child.__class__.__name__
            
            if "Transformer" in module_type or "Attention" in module_type:
                # Wrap inner blocks (FluxTransformerBlock, FluxSingleTransformerBlock)
                if "FluxTransformerBlock" in module_type or "FluxSingleTransformerBlock" in module_type:
                    wrapper = FastCacheOuterWrapper(
                        child,
                        cache_threshold=cache_ratio_threshold,
                        motion_threshold=motion_threshold
                    )
                    setattr(module, name, wrapper)
                    wrapped_blocks.append((full_name, wrapper))
                else:
                    apply_fastcache_recursively(child, full_name)
            else:
                apply_fastcache_recursively(child, full_name)
                
    apply_fastcache_recursively(pipe.transformer)
    print(f"✅ Wrapped {len(wrapped_blocks)} transformer blocks with FastCacheOuterWrapper")

    # -----------------------------
    # C. Safe CPU Offloading
    # -----------------------------
    pipe.enable_model_cpu_offload()

    # -----------------------------
    # D. Prepare Output Targets
    # -----------------------------
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir, exist_ok=True)
    
    with open(csv_file, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(["Prompt", "Seed", "Inference Time (seconds)"])

    prompts = [
        "A highly detailed portrait of a cyberpunk hacker in a neon-lit room, cinematic lighting, intricate facial features, ultra realistic",
        "A sweeping landscape of a distant alien planet with twin suns, dramatic mountains, reflective lakes, volumetric clouds, highly detailed",
        "A massive brutalist concrete building standing alone in a misty forest, architectural photography, sharp geometric lines, overcast lighting",
        "A close-up wildlife photograph of a magnificent snow leopard walking through fresh snow, ultra detailed fur, natural lighting, shallow depth of field",
        "A magical fantasy castle floating on a giant rock above the clouds, waterfalls cascading into the sky, flying dragons, golden sunset, highly detailed concept art"
    ]

    inference_times = []
    total_start_time = time.perf_counter()

    # -----------------------------
    # E. Run inference & Runtime measurements
    # -----------------------------
    for i, prompt in enumerate(prompts):
        print(f"\nFastCache prompt {i + 1}/{len(prompts)}: {prompt}")

        seed = 42 + i
        generator = torch.Generator(device="cpu").manual_seed(seed)
        
        # Reset internal states across generations
        for name, wrapper in wrapped_blocks:
            if hasattr(wrapper, "fastcache"):
                wrapper.fastcache.prev_hidden_states = None
                wrapper.fastcache.bg_hidden_states = None

        start_time = time.perf_counter()

        result = pipe(
            prompt=prompt,
            height=height,
            width=width,
            num_inference_steps=num_inference_steps,
            guidance_scale=guidance_scale,
            generator=generator,
        )

        end_time = time.perf_counter()
        inf_time = end_time - start_time
        inference_times.append(inf_time)

        print(f"⏱️ Inference time: {inf_time:.2f} seconds")

        # Save generated image
        image = result.images[0]
        image_path = os.path.join(output_dir, f"official_fastcache_image_{i+1}.png")
        image.save(image_path)
        print(f"✅ Saved image to {image_path}")

        # Save measurement to CSV
        with open(csv_file, mode='a', newline='') as file:
            writer = csv.writer(file)
            writer.writerow([prompt, seed, f"{inf_time:.4f}"])

    total_end_time = time.perf_counter()
    total_runtime = total_end_time - total_start_time

    # Calculate average and std dev
    avg_inf_time = sum(inference_times) / len(inference_times)
    std_inf_time = math.sqrt(sum((x - avg_inf_time)**2 for x in inference_times) / len(inference_times))

    print(f"\n=== FastCache Performance Metrics ===")
    print(f"Average Inference Time: {avg_inf_time:.2f}s")
    print(f"Inference Time Std Dev: {std_inf_time:.2f}s")
    print(f"Total Runtime (including setup): {total_runtime:.2f}s")
    
    # -----------------------------
    # F. Configuration Artifact
    # -----------------------------
    config_data = {
        "model_id": model_id,
        "prompts": prompts,
        "seeds": [42 + i for i in range(len(prompts))],
        "height": height,
        "width": width,
        "inference_steps": num_inference_steps,
        "fastcache_parameters": {
            "cache_ratio_threshold": cache_ratio_threshold,
            "motion_threshold": motion_threshold,
            "significance_level": 0.05
        },
        "quantization": "bitsandbytes_4bit_nf4",
        "cpu_offload": True
    }
    
    with open("official_fastcache_config.json", "w") as f:
        json.dump(config_data, f, indent=4)
        
    print("✅ Configuration successfully saved to official_fastcache_config.json")
    
    # -----------------------------
    # G. Cache Hit Statistics
    # -----------------------------
    total_hits = 0
    total_ops = 0
    for name, wrapper in wrapped_blocks:
        if hasattr(wrapper, "fastcache"):
            total_hits += wrapper.fastcache.cache_hits
            total_ops += wrapper.fastcache.total_steps
    
    print(f"\n=== Cache Statistics ===")
    if total_ops > 0:
        print(f"Overall Cache Hit Ratio: {total_hits}/{total_ops} ({total_hits/total_ops:.2%})")
    else:
        print("No cache operations recorded.")


run_official_fastcache()
